In [ ]:
import duckdb

from dotenv import dotenv_values

config = dotenv_values(".env")

In [ ]:
datum = '2026-03-01'

In [ ]:
duck = duckdb.connect()

In [ ]:
duck.sql(f"""INSTALL postgres;
                            LOAD postgres;
                            ATTACH 'dbname=zvbn_postgis user={config['POSTGRES_USER']} 
                            host=127.0.0.1 password={config['POSTGRES_PW']}' AS db_dm (TYPE POSTGRES, READ_ONLY);
                            """
                        )

In [ ]:
duck.sql("""ATTACH '/home/zvbn/python/ivu/db/ivu_rt2.db' AS ivu;""")

In [ ]:
duck.sql("describe ivu.rt").df()

## Vergleich der Verläufe nach Nummern

In [ ]:
datum = '2026-05-26'

In [ ]:
duck.sql(f"""select distinct linie, kurs::int
         from ivu.rt 
         where datum = '{datum}'
         and linie in (6352,1270)
         order by linie, kurs
         """).df()

In [ ]:
df = duck.sql(f"""select kurs::int as kurs, nr, hpkt, haltestelle_name
         from ivu.rt 
         where 
         datum = '{datum}'
         -- and linie in (1270)
         -- and kurs in (1226005, 1102202, 1102308)
         and kurs in (6352017, 1270013)
         and nr < 18

         -- and hpkt in (6002502)
         order by  kurs, nr
         """).df()
df
df.to_excel('reports/mastzuordnung_lappan.xlsx', index=False)

In [ ]:
duck.sql("""
         select hpkt, count(*) anz from (
         select distinct hpkt, haltestelle_name
         from ivu.rt where datum > '2026-05-01' 
         group by all
         order by hpkt)
         group by hpkt
         order by anz desc
         """).df()

In [ ]:
duck.sql("""
         select hpkt, count(*) anz from (
         select distinct hpkt, haltestelle_name
         from ivu.rt where datum > '2026-05-01' 
         group by all
         order by hpkt)
         group by hpkt
         order by anz desc
         """).df()

In [ ]:
duck.sql("""select distinct hpkt, string_agg(distinct haltestelle_name, '#' order by haltestelle_name) as haltestellen, 
         string_agg(distinct linie, '#' order by linie) as linien
         from ivu.rt 
         where datum > '2026-05-01' 
         -- and hpkt = 1750902
         group by all
         order by hpkt """).df().to_excel('reports/haltestellen_und_linien.xlsx', index=False)

In [ ]:
duck.sql(f"""create or replace table fahrten as 
         select * 
         from read_parquet('/home/zvbn/python/rt2/out/parquet/prod/fahrten*.parquet')
         where datum >= '{datum}';
         """)

In [ ]:
duck.sql(f"""create or replace table verlauf as 
         select * 
         from read_parquet('/home/zvbn/python/rt2/out/parquet/prod/verlauf_2025_01*.parquet')
         where operday = '2025-01-08'
         and lineshortname in ('330');
         """)

In [ ]:
duck.sql("select distinct deviceid from verlauf")

In [ ]:
duck.sql("select * from verlauf").df().to_excel("/home/zvbn/python/rt2/reports/verlauf_2025_01_08.xlsx", index=False)

In [ ]:
duck.sql("describe db_dm.basis.linien").df()

In [ ]:
duck.sql("""create or replace table linien as 
         select nummer, buendel, dlid, rbl_li_nr from  db_dm.basis.linien""")

In [ ]:
duck.sql("select distinct lineid_short from fahrten order by lineid_short").df()

In [ ]:
duck.sql("show tables from ivu.main;").df()

In [ ]:
duck.sql("describe ivu.main.rt;").df()

In [ ]:
duck.sql(f"""create or replace table fahrten_ivu as 
         select i.datum,i.linie,i.kurs,i.polizeiliches_kennzeichen, l.*
         from ivu.main.rt i
         left join linien l on i.linie = l.rbl_li_nr 
            where i.datum >= '{datum}'

            group by all
         ---limit 10;
         """)

In [ ]:
duck.sql("from fahrten_ivu")

In [ ]:
duck.sql("""select distinct linie 
         from fahrten_ivu
         where linie::text like '_3__'
         order by linie""").df()

## Abfrage einzelner Kennzeichen

In [ ]:
duck.sql("""select datum, kurs, polizeiliches_kennzeichen
         from ivu.main.rt 
         where polizeiliches_kennzeichen like 'DH-R%3324%'
         order by datum desc
         --- limit 10
         """).df()

## Abfrage des Fahrtverlaufs aus IVU.control

In [ ]:
duck.sql("""select datum, linie, kurs::int, polizeiliches_kennzeichen, nr, sollab, istab
         from ivu.main.rt 
         where datum = '2026-05-26' 
         and kurs in (1340113, 1340015)
         order by datum, kurs, nr
         --- limit 10
         """).df()

In [ ]:
duck.sql("select distinct buendel from fahrten_ivu").df()

In [ ]:
duck.sql("""select * 
         from fahrten f
         left join fahrten_ivu i
         on 
            f.datum = i.datum and 
            f.fnr::text = i.kurs::int::text and
            f.lineid_short = i.dlid
         
         where i.buendel = 'AM Ost'
         and f.hasrealtime = false
         order by f.datum, f.fnr
         """).df().to_excel('reports/ohne_echtzeit_amm_ost.xlsx', index=False)

In [ ]:
duck.close()